In [ ]:
!pip install -q transformers sentencepiece x-transformers optuna scikit-learn

In [ ]:
import os
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from x_transformers import Decoder

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

import optuna

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", device)

In [ ]:
MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"

MAX_LEN = 128

EPOCHS = 100

BATCH_SIZE = 64

NUM_LABELS = 2

In [ ]:
TRAIN_PATH = "/content/train.json"

VALID_PATH = "/content/valid.json"

TEST_PATH = "/content/test.json"

In [ ]:
def load_json_dataset(path):

    with open(path, "r", encoding="utf-8") as f:

        raw_data = json.load(f)

    texts = []

    labels = []

    for item in raw_data:

        if len(item) != 2:
            continue

        text, label = item

        if not isinstance(text, str):
            continue

        # =================================================
        # CLEAN TEXT
        # =================================================
        text = text.replace(
            "---- “ ----",
            " "
        )

        text = text.replace(
            "\n",
            " "
        )

        text = " ".join(
            text.split()
        )

        text = text.strip()

        if len(text) < 2:
            continue

        # =================================================
        # LABEL
        # =================================================
        if label == "depression":

            label_id = 1

        elif label == "no_depression":

            label_id = 0

        else:
            continue

        texts.append(text)

        labels.append(label_id)

    df = pd.DataFrame({
        "text": texts,
        "label": labels
    })

    return df


In [ ]:
train_df = load_json_dataset(TRAIN_PATH)

val_df = load_json_dataset(VALID_PATH)

test_df = load_json_dataset(TEST_PATH)

In [ ]:
print("\nTRAIN:", len(train_df))

print("VALID:", len(val_df))

print("TEST :", len(test_df))

print("\nTRAIN LABEL COUNTS")
print(train_df["label"].value_counts())

print("\nVALID LABEL COUNTS")
print(val_df["label"].value_counts())

print("\nTEST LABEL COUNTS")
print(test_df["label"].value_counts())

In [ ]:
TEXT_COLUMN = "text"

LABEL_COLUMN = "label"

LABEL_NAMES = [
    "no_depression",
    "depression"
]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [ ]:
class EmotionDataset(Dataset):

    def __init__(self, df):

        self.texts = df[TEXT_COLUMN].tolist()

        self.labels = df[LABEL_COLUMN].tolist()

    def __len__(self):

        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        label = self.labels[idx]

        encoding = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {

            "input_ids":
                encoding["input_ids"].squeeze(0),

            "attention_mask":
                encoding["attention_mask"].squeeze(0),

            "labels":
                torch.tensor(
                    label,
                    dtype=torch.long
                )
        }


In [ ]:
train_dataset = EmotionDataset(train_df)

val_dataset = EmotionDataset(val_df)

test_dataset = EmotionDataset(test_df)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [ ]:
class_counts = (
    train_df[LABEL_COLUMN]
    .value_counts()
    .sort_index()
    .values
)

class_weights = 1.0 / class_counts

class_weights = (
    class_weights /
    class_weights.sum()
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
).to(device)

print("\nCLASS WEIGHTS:")
print(class_weights)


In [ ]:
class AttentionPooling(nn.Module):

    def __init__(self, hidden_size):

        super().__init__()

        self.attention = nn.Sequential(

            nn.Linear(
                hidden_size,
                hidden_size
            ),

            nn.Tanh(),

            nn.Linear(
                hidden_size,
                1
            )
        )

    def forward(self, x, mask):

        scores = self.attention(x).squeeze(-1)

        scores = scores.masked_fill(
            mask == 0,
            -1e9
        )

        weights = torch.softmax(
            scores,
            dim=1
        )

        pooled = torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )

        return pooled


In [ ]:
class EmotionModel(nn.Module):

    def __init__(
        self,
        depth=1,
        heads=2,
        attn_dropout=0.2,
        ff_dropout=0.2,
        ff_mult=2
    ):

        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME
        )

        hidden_size = (
            self.encoder
            .config
            .hidden_size
        )

        self.decoder = Decoder(

            dim=hidden_size,

            depth=depth,

            heads=heads,

            attn_dropout=attn_dropout,

            ff_dropout=ff_dropout,

            ff_mult=ff_mult
        )

        self.pooling = AttentionPooling(
            hidden_size
        )

        self.dropout = nn.Dropout(0.4)

        self.fc = nn.Linear(
            hidden_size,
            NUM_LABELS
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        x = outputs.last_hidden_state

        x = self.decoder(x)

        x = self.pooling(
            x,
            attention_mask
        )

        x = self.dropout(x)

        logits = self.fc(x)

        return logits

In [ ]:
def evaluate(model, loader):

    model.eval()

    all_labels = []

    all_preds = []

    with torch.no_grad():

        for batch in loader:

            input_ids = (
                batch["input_ids"]
                .to(device)
            )

            attention_mask = (
                batch["attention_mask"]
                .to(device)
            )

            labels = (
                batch["labels"]
                .to(device)
            )

            logits = model(
                input_ids,
                attention_mask
            )

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    weighted_f1 = f1_score(
        all_labels,
        all_preds,
        average="weighted"
    )

    return {

        "accuracy":
            accuracy,

        "macro_f1":
            macro_f1,

        "weighted_f1":
            weighted_f1,

        "labels":
            np.array(all_labels),

        "preds":
            np.array(all_preds)
    }

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    lr,
    weight_decay
):

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    total_steps = (
        len(train_loader)
        * EPOCHS
    )

    scheduler = (
        get_linear_schedule_with_warmup(

            optimizer,

            num_warmup_steps=int(
                0.1 * total_steps
            ),

            num_training_steps=total_steps
        )
    )

    scaler = torch.cuda.amp.GradScaler()

    best_macro_f1 = 0

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        loop = tqdm(train_loader)

        for batch in loop:

            input_ids = (
                batch["input_ids"]
                .to(device)
            )

            attention_mask = (
                batch["attention_mask"]
                .to(device)
            )

            labels = (
                batch["labels"]
                .to(device)
            )

            optimizer.zero_grad()

            with torch.cuda.amp.autocast():

                logits = model(
                    input_ids,
                    attention_mask
                )

                loss = criterion(
                    logits,
                    labels
                )

            scaler.scale(loss).backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            scaler.step(optimizer)

            scaler.update()

            scheduler.step()

            total_loss += loss.item()

            loop.set_description(
                f"Epoch {epoch+1}/{EPOCHS}"
            )

            loop.set_postfix(
                loss=loss.item()
            )

        val_result = evaluate(
            model,
            val_loader
        )

        print("\n======================")

        print(f"Epoch {epoch+1}")

        print("======================")

        print(
            f"VAL ACCURACY   : "
            f"{val_result['accuracy']:.4f}"
        )

        print(
            f"VAL MACRO F1   : "
            f"{val_result['macro_f1']:.4f}"
        )

        print(
            f"VAL WEIGHTED F1: "
            f"{val_result['weighted_f1']:.4f}"
        )

        if (
            val_result["macro_f1"]
            > best_macro_f1
        ):

            best_macro_f1 = (
                val_result["macro_f1"]
            )

            torch.save(
                model.state_dict(),
                "/content/best_model.pt"
            )

    model.load_state_dict(
        torch.load(
            "/content/best_model.pt"
        )
    )

    return model

In [ ]:
def objective(trial):

    depth = trial.suggest_int(
        "depth",
        1,
        2
    )

    heads = trial.suggest_categorical(
        "heads",
        [2, 4]
    )

    attn_dropout = trial.suggest_float(
        "attn_dropout",
        0.2,
        0.5
    )

    ff_dropout = trial.suggest_float(
        "ff_dropout",
        0.2,
        0.5
    )

    ff_mult = trial.suggest_int(
        "ff_mult",
        2,
        4
    )

    lr = trial.suggest_float(
        "lr",
        1e-5,
        2e-4,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-4,
        1e-2,
        log=True
    )

    model = EmotionModel(

        depth=depth,

        heads=heads,

        attn_dropout=attn_dropout,

        ff_dropout=ff_dropout,

        ff_mult=ff_mult

    ).to(device)

    model = train_model(
        model,
        train_loader,
        val_loader,
        lr,
        weight_decay
    )

    val_result = evaluate(
        model,
        val_loader
    )

    return val_result["macro_f1"]

In [ ]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=10
)

# =========================================================
# BEST PARAMETERS
# =========================================================
print("\n======================")

print("BEST PARAMETERS")

print("======================")

print(study.best_params)

best_params = study.best_params

In [ ]:
final_model = EmotionModel(

    depth=best_params["depth"],

    heads=best_params["heads"],

    attn_dropout=best_params["attn_dropout"],

    ff_dropout=best_params["ff_dropout"],

    ff_mult=best_params["ff_mult"]

).to(device)

In [ ]:
final_model = train_model(

    final_model,

    train_loader,

    val_loader,

    best_params["lr"],

    best_params["weight_decay"]
)

In [ ]:
test_result = evaluate(
    final_model,
    test_loader
)

print("\n======================")

print("FINAL RESULT")

print("======================")

print(
    f"Accuracy     : "
    f"{test_result['accuracy']:.4f}"
)

print(
    f"Macro F1     : "
    f"{test_result['macro_f1']:.4f}"
)

print(
    f"Weighted F1  : "
    f"{test_result['weighted_f1']:.4f}"
)

In [ ]:
print("\n======================")

print("CLASSIFICATION REPORT")

print("======================")

print(
    classification_report(

        test_result["labels"],

        test_result["preds"],

        target_names=LABEL_NAMES,

        zero_division=0
    )
)

In [ ]:
print("\n======================")

print("CONFUSION MATRIX")

print("======================")

cm = confusion_matrix(
    test_result["labels"],
    test_result["preds"]
)

print(cm)

In [ ]:
AVE_PATH = (
    "/content/"
    "final_depression_model.pt"
)

torch.save(
    {

        "model_state_dict":
            final_model.state_dict(),

        "best_params":
            best_params,

        "labels":
            LABEL_NAMES

    },
    SAVE_PATH
)

print("\nMODEL SAVED:", SAVE_PATH)